# 40 — Self-Healing CI Agent

A real agentic repair loop powered by OpenAI function calling. The agent reads a CI failure log, calls tools like `apply_dependency_fix` and `run_tests`, observes their results, and loops until the tests pass or all retries are exhausted. No fake structured-output patches — every tool call produces an actual observable outcome.


## Pattern — real tool-calling vs structured-output simulation

| Approach | How repair is decided | How outcome is observed |
|----------|----------------------|------------------------|
| Old (structured output) | LLM writes text describing what it *would* do | LLM validates its own proposal |
| New (tool calling) | LLM calls `apply_dependency_fix(package='requests')` | `run_tests()` returns `passed=true` |

The new loop is grounded: the LLM cannot claim success without the tool returning `passed=true`.


In [ ]:
!pip install openai pydantic python-dotenv


In [ ]:
import os

# Replace with your actual OpenAI API key before running
os.environ["OPENAI_API_KEY"] = "sk-..."


## Schema


In [ ]:
from __future__ import annotations  # noqa: F404

from pydantic import BaseModel, Field


class CIFailure(BaseModel):
    """Input: a single CI job failure to be healed."""

    error_log: str = Field(
        description=(
            "Full error log from the CI job (stderr, traceback, exit output). "
            "This is passed verbatim to the agent as the problem to fix."
        )
    )
    job_name: str = Field(
        description="Name of the CI job that failed (e.g. 'build', 'test', 'lint')."
    )


class RepairAttempt(BaseModel):
    """Record of a single tool call made during the repair loop."""

    iteration: int = Field(
        description="1-based loop iteration in which this tool call occurred."
    )
    tool_called: str = Field(
        description=(
            "Name of the tool the agent invoked "
            "(apply_dependency_fix | apply_env_fix | apply_code_patch | run_tests)."
        )
    )
    arguments: dict = Field(
        description="Arguments passed to the tool, as a dict."
    )
    result: dict = Field(
        description="Return value from the tool, as a dict."
    )


class RepairPostmortem(BaseModel):
    """Terminal postmortem emitted when all retries are exhausted."""

    job_name: str = Field(
        description="Name of the CI job that could not be healed."
    )
    terminal_failure: bool = Field(
        description=(
            "Always True for a postmortem — indicates the agent could not resolve "
            "the issue."
        )
    )
    root_cause: str = Field(
        description="Final assessment of the root cause after all repair attempts."
    )
    recommended_fix: str = Field(
        description="Best recommended fix for a human engineer to apply manually."
    )
    escalation_notes: str = Field(
        description=(
            "Context and next steps for the on-call engineer or team receiving "
            "this escalation."
        )
    )


class HealingResult(BaseModel):
    """Top-level result returned by workflow.run()."""

    healed: bool = Field(
        description=(
            "True if the agent successfully healed the failure, False if retries "
            "were exhausted."
        )
    )
    iterations_used: int = Field(
        description="Total number of loop iterations consumed."
    )
    attempts: list[RepairAttempt] = Field(
        description="All tool calls made during the healing run, in order."
    )
    postmortem: RepairPostmortem | None = Field(
        default=None,
        description="Structured postmortem, populated only when healed=False.",
    )


## System prompts


In [ ]:
HEAL_SYSTEM = """
You are a self-healing CI agent. Your job is to diagnose and fix CI failures
by calling real tools and observing their results.

You have four tools available:
  - apply_dependency_fix(package, version?)
      Install or upgrade a Python package.
  - apply_env_fix(key, value)
      Set an environment variable.
  - apply_code_patch(file_path, description)
      Apply a code change to a file.
  - run_tests(filter_pattern?)
      Run the CI test suite. Always call this after a fix.

Rules:
1. Apply one fix at a time, then call run_tests.
2. Stop immediately when run_tests returns passed=true.
3. If you have no remaining strategies, stop calling tools.
""".strip()

POSTMORTEM_SYSTEM = """
You are a senior reliability engineer writing a terminal postmortem after a
self-healing CI agent exhausted all repair attempts without resolving a failure.

Write a RepairPostmortem (JSON only) with fields:
  job_name, terminal_failure (true), root_cause, recommended_fix, escalation_notes
""".strip()


## Tool implementations (simulated)


In [ ]:
# ---------------------------------------------------------------------------
# Simulation state — reset per run() call
# ---------------------------------------------------------------------------
_applied_fixes: list[dict] = []


def reset_state() -> None:
    global _applied_fixes
    _applied_fixes = []


def apply_dependency_fix(package: str, version: str | None = None) -> dict:
    """Install or upgrade a Python package."""
    fix = {"type": "dependency", "package": package, "version": version}
    _applied_fixes.append(fix)
    ver_str = f"=={version}" if version else " (latest)"
    return {
        "success": True,
        "message": f"Installed {package}{ver_str}",
        "fix": fix,
    }


def apply_env_fix(key: str, value: str) -> dict:
    """Set an environment variable in the CI context."""
    fix = {"type": "env", "key": key, "value": value}
    _applied_fixes.append(fix)
    return {
        "success": True,
        "message": f"Set {key}={value}",
        "fix": fix,
    }


def apply_code_patch(file_path: str, description: str) -> dict:
    """Apply a described code change to a file."""
    fix = {"type": "code_patch", "file": file_path, "description": description}
    _applied_fixes.append(fix)
    return {
        "success": True,
        "message": f"Patched {file_path}: {description}",
        "fix": fix,
    }


def run_tests(filter_pattern: str | None = None) -> dict:
    """Run the CI test suite. Passes if a dependency fix was applied."""
    dep_fixes = [f for f in _applied_fixes if f["type"] == "dependency"]
    if dep_fixes:
        pattern_note = f" (filter: {filter_pattern})" if filter_pattern else ""
        return {
            "passed": True,
            "exit_code": 0,
            "output": (
                f"pytest{pattern_note}: 42 passed in 3.2s "
                f"(after installing {dep_fixes[-1]['package']})"
            ),
        }
    return {
        "passed": False,
        "exit_code": 1,
        "output": "ERROR: Connection timed out after 30s. Tests did not complete.",
    }


TOOL_DEFINITIONS: list[dict] = [
    {
        "type": "function",
        "function": {
            "name": "apply_dependency_fix",
            "description": "Install or upgrade a Python package.",
            "parameters": {
                "type": "object",
                "properties": {
                    "package": {"type": "string"},
                    "version": {"type": "string"},
                },
                "required": ["package"],
                "additionalProperties": False,
            },
            "strict": True,
        },
    },
    {
        "type": "function",
        "function": {
            "name": "apply_env_fix",
            "description": "Set an environment variable in the CI context.",
            "parameters": {
                "type": "object",
                "properties": {
                    "key": {"type": "string"},
                    "value": {"type": "string"},
                },
                "required": ["key", "value"],
                "additionalProperties": False,
            },
            "strict": True,
        },
    },
    {
        "type": "function",
        "function": {
            "name": "apply_code_patch",
            "description": "Apply a described code change to a source file.",
            "parameters": {
                "type": "object",
                "properties": {
                    "file_path": {"type": "string"},
                    "description": {"type": "string"},
                },
                "required": ["file_path", "description"],
                "additionalProperties": False,
            },
            "strict": True,
        },
    },
    {
        "type": "function",
        "function": {
            "name": "run_tests",
            "description": (
                "Run the CI test suite. "
                "Stop calling tools if this returns passed=true."
            ),
            "parameters": {
                "type": "object",
                "properties": {
                    "filter_pattern": {"type": "string"},
                },
                "required": [],
                "additionalProperties": False,
            },
            "strict": True,
        },
    },
]

TOOL_MAP = {
    "apply_dependency_fix": apply_dependency_fix,
    "apply_env_fix": apply_env_fix,
    "apply_code_patch": apply_code_patch,
    "run_tests": run_tests,
}


## Tool-calling repair loop


In [ ]:
import json
import os

from openai import OpenAI

_MODEL = "gpt-4o-mini"


def _get_client() -> OpenAI:
    return OpenAI(api_key=os.environ["OPENAI_API_KEY"])


def _write_postmortem(
    failure: CIFailure,
    attempts: list[RepairAttempt],
) -> RepairPostmortem:
    """Generate a structured postmortem after all iterations are exhausted."""
    client = _get_client()
    user_content = json.dumps(
        {
            "job_name": failure.job_name,
            "error_log": failure.error_log,
            "tool_calls_made": [a.model_dump() for a in attempts],
        },
        indent=2,
    )
    resp = client.chat.completions.create(
        model=_MODEL,
        messages=[
            {"role": "system", "content": POSTMORTEM_SYSTEM},
            {"role": "user", "content": user_content},
        ],
        response_format={
            "type": "json_schema",
            "json_schema": {
                "name": "RepairPostmortem",
                "strict": True,
                "schema": RepairPostmortem.model_json_schema(),
            },
        },
    )
    return RepairPostmortem.model_validate_json(resp.choices[0].message.content)


def run(failure: CIFailure, max_iterations: int = 8) -> HealingResult:
    """Real tool-calling self-healing loop for a CI failure."""
    reset_state()
    client = _get_client()

    messages: list[dict] = [
        {"role": "system", "content": HEAL_SYSTEM},
        {
            "role": "user",
            "content": (
                f"CI job: {failure.job_name}\n\n"
                f"Failure log:\n\n{failure.error_log}\n\n"
                "Fix it using the available tools."
            ),
        },
    ]

    attempts: list[RepairAttempt] = []

    for i in range(max_iterations):
        resp = client.chat.completions.create(
            model=_MODEL,
            messages=messages,
            tools=TOOL_DEFINITIONS,
            tool_choice="auto",
        )
        msg = resp.choices[0].message

        assistant_msg: dict = {"role": "assistant", "content": msg.content or ""}
        if msg.tool_calls:
            assistant_msg["tool_calls"] = [
                {
                    "id": tc.id,
                    "type": "function",
                    "function": {
                        "name": tc.function.name,
                        "arguments": tc.function.arguments,
                    },
                }
                for tc in msg.tool_calls
            ]
        messages.append(assistant_msg)

        if not msg.tool_calls:
            break

        for tc in msg.tool_calls:
            fn = TOOL_MAP.get(tc.function.name)
            if fn is None:
                result = {"error": f"Unknown tool: {tc.function.name}"}
            else:
                args = json.loads(tc.function.arguments)
                result = fn(**args)

            messages.append(
                {
                    "role": "tool",
                    "tool_call_id": tc.id,
                    "content": json.dumps(result),
                }
            )

            attempt = RepairAttempt(
                iteration=i + 1,
                tool_called=tc.function.name,
                arguments=json.loads(tc.function.arguments),
                result=result,
            )
            attempts.append(attempt)

            if tc.function.name == "run_tests" and result.get("passed"):
                return HealingResult(
                    healed=True,
                    iterations_used=i + 1,
                    attempts=attempts,
                    postmortem=None,
                )

    postmortem = _write_postmortem(failure, attempts)
    return HealingResult(
        healed=False,
        iterations_used=len(attempts),
        attempts=attempts,
        postmortem=postmortem,
    )


## Scenario A — missing dependency (expected to heal)


In [ ]:
SCENARIO_A = CIFailure(
    job_name="build",
    error_log=(
        "Collecting dependencies from requirements.txt\n"
        "ERROR: Could not find a version that satisfies the requirement requests>=2.28.0\n"
        "ERROR: No matching distribution found for requests>=2.28.0\n"
        "ModuleNotFoundError: No module named 'requests'\n"
        "Command 'pip install -r requirements.txt' returned non-zero exit status 1."
    ),
)

result_a = run(SCENARIO_A, max_iterations=8)
print(f'Healed:          {result_a.healed}')
print(f'Iterations used: {result_a.iterations_used}')
print(f'Tool calls made: {len(result_a.attempts)}')
for attempt in result_a.attempts:
    status = ''
    if attempt.tool_called == 'run_tests':
        status = ' [PASSED]' if attempt.result.get('passed') else ' [FAILED]'
    args_str = ', '.join(f'{k}={v!r}' for k, v in attempt.arguments.items())
    print(f'  [iter {attempt.iteration}] {attempt.tool_called}({args_str}){status}')


## Scenario B — flaky network timeout (expected to exhaust retries + postmortem)


In [ ]:
SCENARIO_B = CIFailure(
    job_name="test",
    error_log=(
        "FAILED tests/test_payment_processor.py::test_stripe_charge_idempotency\n"
        "AssertionError: assert response.status_code == 200\n"
        "E   requests.exceptions.ConnectionError: "
        "HTTPSConnectionPool(host='api.stripe.com'): "
        "Max retries exceeded with url: /v1/charges\n"
        "1 failed, 47 passed in 12.34s\n"
        "This test fails intermittently on CI runners due to network timeouts. "
        "Retry count: 3/3 exhausted."
    ),
)

result_b = run(SCENARIO_B, max_iterations=8)
print(f'Healed:          {result_b.healed}')
print(f'Iterations used: {result_b.iterations_used}')
print(f'Tool calls made: {len(result_b.attempts)}')
if result_b.postmortem:
    pm = result_b.postmortem
    print('\n--- Postmortem ---')
    print(f'Root cause:      {pm.root_cause}')
    print(f'Recommended fix: {pm.recommended_fix}')
    print(f'Escalation:      {pm.escalation_notes}')


## Starter exercise

Add a **token budget gate**: track cumulative `total_tokens` from `resp.usage.total_tokens` across every API call and halt early if the budget is exceeded, emitting a postmortem. What field would you add to `HealingResult` and where in the loop would you check it?


In [ ]:
# Answer key — token budget gate

# 1. Extend the schema:
class HealingResultWithBudget(HealingResult):
    """HealingResult extended with token accounting."""

    total_tokens_used: int = Field(
        default=0,
        description="Cumulative tokens across all LLM calls in the run.",
    )
    budget_exceeded: bool = Field(
        default=False,
        description="True when the loop halted early due to token budget.",
    )


# 2. Track tokens in the loop:
def run_with_budget(
    failure: CIFailure,
    max_iterations: int = 8,
    token_budget: int = 5000,
) -> HealingResultWithBudget:
    """Tool-calling loop with a token-cost gate."""
    reset_state()
    client = _get_client()
    messages: list[dict] = [
        {"role": "system", "content": HEAL_SYSTEM},
        {
            "role": "user",
            "content": f"CI job: {failure.job_name}\n\n{failure.error_log}",
        },
    ]
    attempts: list[RepairAttempt] = []
    total_tokens = 0

    for i in range(max_iterations):
        if total_tokens >= token_budget:
            pm = _write_postmortem(failure, attempts)
            return HealingResultWithBudget(
                healed=False, iterations_used=i, attempts=attempts,
                postmortem=pm, total_tokens_used=total_tokens, budget_exceeded=True,
            )

        resp = client.chat.completions.create(
            model=_MODEL, messages=messages,
            tools=TOOL_DEFINITIONS, tool_choice='auto',
        )
        total_tokens += resp.usage.total_tokens if resp.usage else 0
        msg = resp.choices[0].message

        assistant_msg: dict = {"role": "assistant", "content": msg.content or ""}
        if msg.tool_calls:
            assistant_msg["tool_calls"] = [
                {"id": tc.id, "type": "function",
                 "function": {"name": tc.function.name, "arguments": tc.function.arguments}}
                for tc in msg.tool_calls
            ]
        messages.append(assistant_msg)

        if not msg.tool_calls:
            break

        for tc in msg.tool_calls:
            fn = TOOL_MAP.get(tc.function.name)
            result = fn(**json.loads(tc.function.arguments)) if fn else {"error": "unknown"}
            messages.append({"role": "tool", "tool_call_id": tc.id, "content": json.dumps(result)})
            attempts.append(RepairAttempt(
                iteration=i + 1, tool_called=tc.function.name,
                arguments=json.loads(tc.function.arguments), result=result,
            ))
            if tc.function.name == "run_tests" and result.get("passed"):
                return HealingResultWithBudget(
                    healed=True, iterations_used=i + 1, attempts=attempts,
                    postmortem=None, total_tokens_used=total_tokens, budget_exceeded=False,
                )

    pm = _write_postmortem(failure, attempts)
    return HealingResultWithBudget(
        healed=False, iterations_used=len(attempts), attempts=attempts,
        postmortem=pm, total_tokens_used=total_tokens, budget_exceeded=False,
    )


# Demo with a tight budget to trigger early halt on Scenario B
result_budget = run_with_budget(SCENARIO_B, max_iterations=8, token_budget=300)
print(f'Healed:          {result_budget.healed}')
print(f'Budget exceeded: {result_budget.budget_exceeded}')
print(f'Tokens used:     {result_budget.total_tokens_used}')
print(f'Attempts taken:  {result_budget.iterations_used}')
